In [47]:
import numpy as np 
from ase import Atoms
from ase.io import write
from gpaw import GPAW, PW
from ase.optimize import BFGS

## Density Functional Theory

#### PBC

In [48]:
L = 10.0 

au2 = Atoms(
    'Au2',
    positions=[
        [0, 0, 0],
        [L/2, L/2, L/2]
    ],
    cell=[L, L, L],
    pbc=True
)

au2.calc = GPAW(
    mode=PW(600), # --- Basis Set, 600 eV cutoff
    xc='PBE', # --- Exchange–Correlation (XC) Functional
    kpts=(1,1,1), # --- k-points Sampling
    occupations={'name': 'fermi-dirac', 'width': 0.1}, # --- Occupations / Smearing
    convergence={'energy': 1e-5}, # --- SCF Convergence Settings
    txt='../_files_/dft_au2_print.log'
)

opt = BFGS(au2, trajectory='../_files_/dft_au2.traj', logfile='../_files_/dft_au2.log')
opt.run(fmax=0.05)

# --- Final energy
energy = au2.get_potential_energy()
print("Final Energy:", energy)

# --- Save wavefunction
au2.calc.write('../_files_/dft_au2.gpw')

# --- Save optimized structure
write('../_files_/dft_au2.cif', au2)

Final Energy: -0.12663440880046398


In [49]:
for atom in au2:
    s = atom.symbol
    x,y,z = atom.position
    print(f'{s} {x:6.2f} {y:6.2f} {z:6.2f}')

Au   0.00   0.00   0.00
Au   5.00   5.00   5.00


#### Molecule

In [50]:
d = 0.9575
t = np.pi / 180 * 104.51
h2o = Atoms(
    'H2O',
    positions=[
        (d, 0, 0),
        (d * np.cos(t), d * np.sin(t), 0),
        (0, 0, 0)
    ]
)
h2o.center(vacuum=6.0)
h2o.pbc = False

h2o.calc = GPAW(
    mode='lcao',
    basis='dzp', # Numeric Atomic Orbital (NAO)
    xc='PBE',
    convergence={'energy': 1e-5},
    txt='../_files_/dft_h2o_print.log'
)

opt = BFGS(h2o, trajectory='../_files_/dft_h2o.traj', logfile='../_files_/dft_h2o.log')
opt.run(fmax=0.05)

energy = h2o.get_potential_energy()
print("Final Energy:", energy)

h2o.calc.write('../_files_/dft_h2o.gpw')
write('../_files_/dft_h2o.xyz', h2o)

Final Energy: -13.027383588400223
